In [ ]:
# ═══ 0. Colab：拉取代码 + 装依赖（本地自动跳过）═══
from __future__ import annotations

import io
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path
from urllib.request import Request, urlopen

REPO_OWNER, REPO_NAME, BRANCH = "Beater-221E", "llm4rec-bias-Integrated", "main"
REPO_DIR = Path("/content/llm4rec-bias-Integrated")
GITHUB_TOKEN = ""
FRESH_CLONE = True

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if not IN_COLAB:
    print("本地环境，跳过。")
else:
    if FRESH_CLONE and REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
        print(f"已删除 {REPO_DIR}")

    if not ((REPO_DIR / "src" / "llm4rec").exists() and (REPO_DIR / "pyproject.toml").exists()):
        zip_url = f"https://codeload.github.com/{REPO_OWNER}/{REPO_NAME}/zip/refs/heads/{BRANCH}"
        headers = {"User-Agent": "llm4rec-colab"}
        if GITHUB_TOKEN:
            headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"
        print(f"下载 {zip_url}")
        with urlopen(Request(zip_url, headers=headers), timeout=120) as r:
            data = r.read()
        with zipfile.ZipFile(io.BytesIO(data)) as zf:
            top = zf.namelist()[0].split("/")[0]
            tmp = Path("/content") / "_llm4rec_unzip"
            if tmp.exists():
                shutil.rmtree(tmp)
            zf.extractall(tmp)
            (tmp / top).rename(REPO_DIR)
            shutil.rmtree(tmp, ignore_errors=True)
        print(f"代码就绪 → {REPO_DIR}")
    else:
        print(f"已有仓库 → {REPO_DIR}")

    os.chdir(REPO_DIR)
    os.environ["LLM4REC_ROOT"] = str(REPO_DIR)
    sys.path.insert(0, str(REPO_DIR / "src"))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "omegaconf", "pyyaml", "rich", "tqdm", "numpy", "pandas", "scikit-learn",
        "transformers", "accelerate", "datasets", "sentence-transformers", "wandb",
    ])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])
    print("依赖安装完成")
    print("★ 首次安装后建议：Runtime → Restart session，然后从 cell 1 开始（跳过 cell 0）")


In [ ]:
# ═══ 1. 全局参数（只需要改这里）═══
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

EXP = "minionerec_qwen05b_amazon"
GPUS = "0"
WANDB_MODE = "disabled"
FORCE_PREPARE = False
DEEPSPEED = ""
STAGES = ""
RESUME_FROM = ""
OVERRIDES = [
    "hardware.precision=bf16",
    "hardware.devices=0",
    "sid.codebook_size=512",
    "sid.rqvae.pca_dim=512",
    # SID 现在按官方流程：报告碰撞后直接落盘，不再强制唯一
]

def _find_root() -> Path:
    cands = []
    if os.environ.get("LLM4REC_ROOT"):
        cands.append(Path(os.environ["LLM4REC_ROOT"]))
    cands += [
        Path.cwd(), Path.cwd().parent,
        Path("/content/llm4rec-bias-Integrated"),
        Path("/home/sheng/proj/llm4rec-bias-Integrated"),
    ]
    for c in cands:
        if c.exists() and (c / "src" / "llm4rec").exists() and (c / "pyproject.toml").exists():
            return c.resolve()
    raise FileNotFoundError("找不到项目根。Colab 请先跑 cell 0。")

ROOT = _find_root()
os.chdir(ROOT)
src = str(ROOT / "src")
sys.path = [p for p in sys.path if "llm4rec" not in p or p == src]
sys.path.insert(0, src)
os.environ.update({
    "LLM4REC_ROOT": str(ROOT),
    "PYTHONPATH": src,
    "CUDA_VISIBLE_DEVICES": GPUS,
    "TOKENIZERS_PARALLELISM": "false",
    "PYTHONUNBUFFERED": "1",
    "WANDB_MODE": WANDB_MODE,
    "WANDB_PROJECT": os.environ.get("WANDB_PROJECT", "llm4rec-bias"),
})
n_gpu = len([g for g in GPUS.split(",") if g.strip()])
print(f"ROOT={ROOT}\nEXP={EXP}  GPUS={GPUS}  WANDB={WANDB_MODE}")
print(f"OVERRIDES={OVERRIDES}")


In [ ]:
# ═══ 2. 环境自检 ═══
print(f"Python {sys.version.split()[0]}")
try:
    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True
    ).strip()
    print("GPU:", smi)
except Exception:
    print("GPU: nvidia-smi 不可用")

try:
    import torch
    print(f"torch {torch.__version__}  cuda={torch.cuda.is_available()}")
except Exception as e:
    print(f"WARN torch 导入失败（pip 后请 Restart session）: {e}")

import llm4rec
print("llm4rec:", list(llm4rec.__path__)[0])


In [ ]:
# ═══ 3. 列出可跑实验 ═══
from llm4rec.cli.main import cmd_list

cmd_list()


In [ ]:
# ═══ 4. 校验配置 ═══
from llm4rec.core.compose import compose, to_dict, validate
from llm4rec.cli.main import _print_plan

cli_overrides = list(OVERRIDES)
if DEEPSPEED:
    cli_overrides.append(f"hardware.deepspeed={DEEPSPEED}")
cfg = validate(to_dict(compose(EXP, cli_overrides)))
_print_plan(cfg)

ROUTE = cfg["experiment"]["route"]
NEEDS_SID = "sid" in cfg
NEEDS_BM25 = str((cfg.get("decoder") or {}).get("name")) == "bm25_query"
print(f"route={ROUTE}  sid={NEEDS_SID}  bm25={NEEDS_BM25}")


In [ ]:
# ═══ 5. 数据准备 · 下载原始数据 ═══
from llm4rec.cli.main import cmd_download_data

cmd_download_data(cfg, force=FORCE_PREPARE)


In [ ]:
# ═══ 6. 数据准备 · 预处理为四件套 ═══
from llm4rec.cli.main import cmd_prepare_data
from llm4rec.data.base import get_adapter

cmd_prepare_data(cfg, force=FORCE_PREPARE)

adapter = get_adapter(cfg)
proc_dir = Path(adapter.processed_dir(cfg))
print("processed:", proc_dir)
stats = proc_dir / "stats.json"
print(stats.read_text()[:1000] if stats.exists() else "(no stats.json)")


In [ ]:
# ═══ 7. 数据准备 · 编码物品 embedding（MiniOneRec）═══
from llm4rec.cli.main import cmd_embed_items

if NEEDS_SID:
    cmd_embed_items(cfg, force=FORCE_PREPARE)
else:
    print(f"route={ROUTE} 不需要 embedding，跳过")


In [ ]:
# ═══ 8. 数据准备 · 构建 Semantic ID（MiniOneRec）═══
from llm4rec.cli.main import cmd_build_sid

if NEEDS_SID:
    cmd_build_sid(cfg, force=FORCE_PREPARE)
else:
    print(f"route={ROUTE} 不用 SID，跳过")


In [ ]:
# ═══ 9. 数据准备 · 构建 BM25（Rec-R1）═══
from llm4rec.cli.main import cmd_build_bm25

if NEEDS_BM25:
    cmd_build_bm25(cfg, force=FORCE_PREPARE)
else:
    print(f"route={ROUTE} 不用 BM25，跳过")


In [ ]:
# ═══ 10. 数据准备 · 产物检查 ═══
def _show(p: Path, name: str, depth: int = 2) -> None:
    print(f"{'✓' if p.exists() else '✗'} {name}: {p}")
    if p.is_dir() and depth:
        for k in sorted(p.iterdir())[:20]:
            print("    ", k.name)

_show(proc_dir, "processed")
_show(ROOT / "artifacts" / "sid", "sid")
_show(ROOT / "artifacts" / "bm25", "bm25")
_show(ROOT / "artifacts" / "embeddings", "embeddings")
print("数据准备完成 ✓")


In [ ]:
# ═══ 11. 组装训练命令 ═══
import random
import shlex

args = ["--config", EXP]
if STAGES:
    args += ["--stages", STAGES]
if RESUME_FROM:
    args += ["--resume-from", RESUME_FROM]
if DEEPSPEED:
    args += [f"hardware.deepspeed={DEEPSPEED}"]
args += list(OVERRIDES)

env = os.environ.copy()
env["PYTHONPATH"] = str(ROOT / "src") + (":" + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")

if n_gpu > 1:
    os.environ.setdefault("NCCL_P2P_DISABLE", "1")
    os.environ.setdefault("NCCL_IB_DISABLE", "1")
    cmd = [
        "torchrun", "--standalone",
        "--nproc_per_node", str(n_gpu),
        "--master_port", str(20000 + random.randint(0, 19999)),
        "-m", "llm4rec.cli.main", "run", *args,
    ]
else:
    cmd = [sys.executable, "-m", "llm4rec.cli.main", "run", *args]

print(" ".join(shlex.quote(c) for c in cmd))


In [ ]:
# ═══ 12. 训练（SFT → eval → RL/DPO → eval）═══
from datetime import datetime

logs = ROOT / "logs"
logs.mkdir(exist_ok=True)
log_path = logs / f"{EXP}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
print("log →", log_path)

with open(log_path, "w") as f:
    p = subprocess.Popen(
        cmd, cwd=str(ROOT), env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    assert p.stdout
    for line in p.stdout:
        print(line, end="")
        f.write(line)
    rc = p.wait()
if rc != 0:
    raise RuntimeError(f"训练失败 exit={rc}  log={log_path}")
print("训练完成 ✓")


In [ ]:
# ═══ 13. 定位 run 目录 ═══
from llm4rec.data.base import get_adapter

adapter = get_adapter(cfg)
run_root = (
    ROOT / "runs" / adapter.dataset_key(cfg) / ROUTE
    / str(cfg["model"]["name"]).replace("/", "_") / f"seed_{cfg['seed']}"
)
if run_root.exists() and any(run_root.iterdir()):
    RUN_DIR = sorted(p for p in run_root.iterdir() if p.is_dir())[-1]
else:
    hits = sorted((ROOT / "runs").rglob("summary.json"))
    if not hits:
        raise FileNotFoundError("没有 runs/*/summary.json，请先完成训练")
    RUN_DIR = hits[-1].parent
print("RUN_DIR =", RUN_DIR)

summary = json.loads((RUN_DIR / "summary.json").read_text())
for k, v in summary.items():
    if isinstance(v, dict) and v.get("checkpoint"):
        print(f"  {k}: {v['checkpoint']}")


In [ ]:
# ═══ 14. bias / eval 指标 + delta ═══
KEYS = [
    "hr@10", "ndcg@10", "hr_ips@10", "ndcg_ips@10",
    "pop_lift@1", "pop_lift@10", "delta_gap",
    "exposure_gini", "coverage@10", "tier_gap",
    "history_copy_rate", "top1_concentration", "valid_rate",
]
eval_dir = RUN_DIR / "eval"
for ef in sorted(eval_dir.glob("eval_*.json")) if eval_dir.exists() else []:
    m = json.loads(ef.read_text()).get("metrics") or {}
    print(f"\n── {ef.name} ──")
    for k in KEYS:
        if k in m:
            v = m[k]
            print(f"  {k:22s} {v:.6f}" if isinstance(v, float) else f"  {k:22s} {v}")

delta_path = eval_dir / "bias_delta.json"
if delta_path.exists():
    delta = json.loads(delta_path.read_text())
    print("\n── bias_delta (last − first) ──")
    for k in KEYS:
        dk = f"delta/{k}"
        if dk in delta:
            print(f"  {dk:28s} {delta[dk]:+.6f}")
else:
    print("\n(尚无 bias_delta.json)")


In [ ]:
# ═══ 15. （可选）只评测已有 checkpoint ═══
EVAL_ONLY_CKPT = ""  # 例: str(RUN_DIR / "rl" / "final")

if not EVAL_ONLY_CKPT:
    print("跳过（EVAL_ONLY_CKPT 为空）")
else:
    ckpt = Path(EVAL_ONLY_CKPT)
    assert ckpt.exists(), ckpt
    eval_args = ["--config", EXP, "--stages", "eval", "--resume-from", str(ckpt), *OVERRIDES]
    eval_cmd = [sys.executable, "-m", "llm4rec.cli.main", "run", *eval_args]
    print(" ".join(shlex.quote(c) for c in eval_cmd))
    rc = subprocess.run(eval_cmd, cwd=str(ROOT), env=env).returncode
    if rc:
        raise RuntimeError(f"eval-only 失败 exit={rc}")
    print("评测完成 ✓")
